# 🎤 Cantor Virtual (SVS) — Colab

Grave a **sua** voz e ouça-a cantando uma partitura (melodia + letra).

**Antes de começar:** menu `Runtime → Change runtime type → GPU (T4)`.

> Escopo: use apenas vozes próprias/consentidas e músicas que você tenha direito de usar
> (partitura sua, domínio público ou licença aberta). Veja `CONSENT.md`.

## 1. Checar a GPU

In [ ]:
!nvidia-smi -L
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Obter o projeto (escolha UMA célula)

**(a) GitHub** — se você fez push do projeto. **(b) Upload .zip** — sem GitHub.
**(c) Google Drive** — se copiou a pasta para o Drive.

In [ ]:
# (a) Via GitHub — ajuste a URL do seu repositório
REPO_URL = 'https://github.com/SEU_USUARIO/virtual-singer.git'
%cd /content
!git clone $REPO_URL virtual-singer
%cd /content/virtual-singer

In [ ]:
# (b) Via upload de .zip do projeto
from google.colab import files
import zipfile, os
%cd /content
up = files.upload()                       # selecione o virtual-singer.zip
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z:
    z.extractall('/content')
# ajuste se a pasta extraída tiver outro nome
%cd /content/virtual-singer

In [ ]:
# (c) Via Google Drive
from google.colab import drive
drive.mount('/content/drive')
# ajuste o caminho onde você colocou o projeto no Drive:
%cp -r '/content/drive/MyDrive/virtual-singer' /content/virtual-singer
%cd /content/virtual-singer

## 3. Dependências leves + demo (✅ já funciona)

Prova o pipeline de partitura e de transcrição automática, sem os motores neurais.

In [ ]:
!pip install -q pretty_midi music21 g2p_en librosa soundfile
import nltk
for pkg in ['averaged_perceptron_tagger_eng', 'cmudict', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
print('deps leves OK')

In [ ]:
# gera a música de demonstração e mostra a partitura -> fonemas
!python scripts/make_demo_song.py
!python -m src.score.build data/songs/demo

### (opcional) Transcrever a SUA voz numa partitura (AMT)
Suba um áudio com você cantando/assobiando a melodia (mono).

In [ ]:
from google.colab import files
up = files.upload()                       # selecione seu áudio (wav/mp3)
audio = list(up.keys())[0]
!python scripts/transcribe.py "$audio" --name minha_musica

## 4. PyTorch + Seed-VC (timbre zero-shot, motor de voz-guia = DSP)

A voz-guia (melodia) é gerada por um **sintetizador DSP embutido** (numpy puro). O timbre
é imposto pelo **Seed-VC**, que é **zero-shot**: **não treina nada** — clona a voz a partir
de um clipe de referência de 1-30 s. Sem fairseq, roda no Python do Colab.

A célula abaixo instala deps, clona o Seed-VC (e baixa seus requisitos) e baixa o repo.
Os checkpoints do Seed-VC baixam sozinhos na 1ª síntese.

In [ ]:
# Colab já vem com torch+CUDA. Instala as deps do projeto.
!pip install -q -r requirements.txt

# Clona o Seed-VC (timbre zero-shot — SEM fairseq, SEM treino) e baixa seus requisitos.
!python scripts/setup_models.py
!pip install -q -r third_party/seed-vc/requirements.txt

## 5. Escolher a voz de referência (Seed-VC é zero-shot — SEM treino!)

Cada voz é uma pasta em `data/voices/<nome>/` com um clipe de **1-30 s**. Não há treino:
o Seed-VC clona o timbre direto do clipe. Troque o `--voice` na seção 6 para **alternar**
entre vozes instantaneamente.

- **(5a) Voz pronta da VocalSet** (CC BY 4.0) — testar sem gravar.
- **(5b) Sua própria voz** — poucos segundos bastam.

In [ ]:
# (5a) Voz pronta da VocalSet — baixa só um cantor como referência (sem treino).
SINGER = 'female1'        # female1..female9, male1..male11
!python scripts/get_sample_voice.py --singer $SINGER --name vocalset_$SINGER --max-files 5

In [ ]:
# (5b) Sua própria voz — suba 1+ arquivos (poucos segundos a 1 min bastam; é zero-shot).
import os
from google.colab import files
VOICE = 'minha_voz'
os.makedirs(f'data/voices/{VOICE}', exist_ok=True)
up = files.upload()
for fname in up:
    os.replace(fname, f'data/voices/{VOICE}/{fname}')
print('referência pronta em data/voices/' + VOICE)

## 6. Sintetizar — voz cantando a partitura (troque `VOICE` para alternar)

In [ ]:
# Troque VOICE para alternar: 'vocalset_female1' (5a) ou 'minha_voz' (5b).
VOICE = 'vocalset_female1'
SONG = 'demo'           # ou 'minha_musica' se você transcreveu/importou
!python -m src.pipeline --song data/songs/$SONG --voice $VOICE --out out/resultado.wav
from IPython.display import Audio
Audio('out/resultado.wav')

## 7. Avatar visual (opcional) — ⛔ NÃO rode no Colab grátis

> **Atenção:** o Colab **gratuito proíbe** ferramentas de animação facial/deepfake (SadTalker)
> e **encerra a sessão** se você clonar/rodar o SadTalker. Por isso ele ficou de fora das
> células acima.

Rode esta etapa **fora do Colab grátis**:
- **Localmente**, se você tiver uma GPU NVIDIA (sem problema de ToS — dados seus, consentidos);
- ou num host que permita (RunPod / Paperspace / Lightning).

Passos (na sua máquina, depois de gerar `out/resultado.wav`):
```bash
python scripts/setup_models.py --with-avatar      # baixa o SadTalker
python scripts/get_face.py --name meu_avatar      # rosto sintético (não-real)
python -m src.pipeline --song data/songs/demo --voice meu_nome \
    --avatar-image data/faces/meu_avatar.jpg --out out/resultado.wav   # gera out/resultado.mp4
```

⚠️ Use rosto sintético/próprio/consentido — nunca pessoa real sem consentimento. Veja `CONSENT.md`.